# Laboratorio 2. Complejidad y búsqueda de hiperparámetros### ISIS2611 · Aprendizaje de Máquina | Caso AlpesPlanck**Integrantes:** (nombres completos y códigos)**Grupo de entrega:** GL2**Fecha límite:** 14 de septiembre, 20:00---> **Cómo usar este esqueleto.** Cada sección contiene una celda de texto que explica *qué* hay que hacer,> *por qué* se hace y *qué* se debe justificar por escrito, seguida de una celda de código vacía con> comentarios `# TODO`. Ninguna celda trae la solución. Antes de escribir código en una sección, conviene> poder responder en voz alta la pregunta de control que aparece al final de cada bloque.> Al terminar, elimina las notas marcadas como *"Nota del esqueleto"*, para que el notebook entregado se> lea como un informe técnico y no como una guía.

## 0. Mapa del laboratorio y trazabilidad con la rúbricaAntes de programar conviene ver el laboratorio completo, porque el peso de la nota no está donde sueleestar la atención. El código de los modelos vale 40 % y el análisis escrito vale 35 %.| Sección del notebook | Actividad del enunciado | Criterio de rúbrica | Peso ||:---|:---|:---|:---:|| 1. Configuración, datos y protocolo de evaluación | Preparación previa | Transversal, soporta todos | 0 % directo || 2. Regresión polinomial con GridSearchCV | Actividad 1 | Criterio 1 | 15 % || 3. Curvas de validación | Actividad 2 | Se evalúa dentro de los criterios 1 y 6 | incluido || 4. Ridge y Lasso | Actividad 3 | Criterio 2 | 15 % || 5. Polinomial regularizado | Actividad 4 | Criterio 3 | 10 % || 6. Tabla comparativa y selección del mejor modelo | Actividad 5 | Criterio 4 | 5 % || 7. Intervalos de confianza por bootstrapping | Actividad 6 | Criterio 5 | 10 % || 8. Análisis de resultados | Sección "Análisis de resultados" | Criterio 6 | 35 % || 9. Video explicativo | Actividad 7 | Criterio 7 | 5 % || 10. Uso de herramientas de IA generativa | Reglas del curso | Criterio 8 | 5 % |**Nota del esqueleto (importante).** La actividad 2, las curvas de validación, no tiene una fila propiaen la rúbrica. Eso no significa que se pueda omitir: es la evidencia gráfica que sostiene varias de laspreguntas del análisis, que sí valen 35 %. Omitirla cuesta puntos por dos vías, en el criterio 1 porjustificación incompleta y en el criterio 6 por respuestas sin soporte.**Segunda observación.** El criterio 6 vale más que cualquier modelo individual. Un notebook con cuatromodelos impecables y respuestas de dos líneas obtiene menos nota que uno con modelos correctos y unanálisis bien argumentado. Reserva tiempo real para la sección 8.

---## 1. Configuración, datos y protocolo de evaluaciónEsta sección no aparece en la rúbrica, pero condiciona la validez de todo lo demás. Si la partición oel esquema de validación están mal planteados, todos los números posteriores quedan invalidados aunqueel código corra sin errores.

### 1.1 Importación de librerías**Qué hacer.** Importar en una sola celda todo lo que se usará en el notebook, agrupado por origen:manipulación de datos, visualización, y los componentes de scikit-learn (partición, validación cruzada,búsqueda de hiperparámetros, curvas de validación, transformadores de preprocesamiento, escaladores,generación de características polinomiales, los tres estimadores lineales y las métricas de regresión).**Por qué en una sola celda.** Un lector, y el profesor, debe poder reconstruir el entorno leyendo unacelda. Las importaciones dispersas a lo largo del notebook dificultan la revisión y suelen producirerrores al reejecutar desde cero.**Qué agregar aquí también.** La fijación de una semilla global y una constante `RANDOM_STATE` que sereutilice en cada partición, en cada `KFold` y en cada remuestreo. Sin semilla fija, los resultadoscambian entre ejecuciones y las conclusiones del análisis dejan de ser verificables.

In [ ]:
# TODO 1.1# - Importar librerías de manipulación de datos y visualización.# - Importar de scikit-learn: partición, validación cruzada, búsqueda en rejilla,#   curva de validación, pipeline, transformador por columnas, imputación,#   escaladores (tres alternativas), características polinomiales,#   codificación de categóricas, los estimadores lineales y las métricas.# - Definir RANDOM_STATE y fijar la semilla.# - Configurar el formato de impresión de pandas y el estilo de las gráficas.

### 1.2 Carga del conjunto de datos limpio del Laboratorio 1**Qué hacer.** Cargar la versión del conjunto de datos que resultó del proceso de limpieza y preparacióndel Laboratorio 1, no el archivo crudo. El enunciado es explícito en este punto: el énfasis de estelaboratorio no está en la exploración ni en el procesamiento.**Qué documentar.** Un resumen breve, de un párrafo, de qué transformaciones vienen heredadas delLaboratorio 1: correcciones de calidad aplicadas, tratamiento de faltantes, variables derivadas ycolumnas descartadas. El evaluador necesita saber sobre qué datos se está modelando sin abrir ellaboratorio anterior.**Verificación mínima antes de seguir.** Dimensiones del conjunto, tipos de dato por columna, conteo defaltantes y estadísticos descriptivos de la variable objetivo `temp_max_manana`. Si algo cambió respectoal Laboratorio 1, hay que detectarlo aquí y no después de entrenar veinte modelos.**Nota del esqueleto, decisión que debes tomar y justificar.** El archivo `Datos_Test_Lab_1.csv` nocontiene la columna `temp_max_manana`. Sin etiquetas no se pueden calcular RMSE, MAE ni R², y por lotanto ese archivo **no puede** ser el conjunto de test sobre el que se hace el bootstrapping de laactividad 6. El conjunto de test con etiquetas debe salir de una partición interna de `Datos_Lab_1.csv`.Deja esta decisión escrita de forma explícita en el notebook.

In [ ]:
# TODO 1.2# - Cargar el conjunto limpio resultante del Laboratorio 1.# - Verificar dimensiones, tipos, faltantes y descriptivos de la variable objetivo.# - Dejar constancia escrita de qué transformaciones vienen heredadas del Lab 1.

### 1.3 Separación de X e y, y partición entrenamiento / test**Qué hacer.** Separar la matriz de características `X` de la variable objetivo `y` y realizar lapartición en entrenamiento y test **antes de cualquier transformación**. El conjunto de test se guarda yno se vuelve a tocar hasta la sección 7.**Por qué este orden.** Escalar, imputar o expandir antes de partir es fuga de datos: las estadísticasdel preprocesamiento habrían visto las filas de test. El curso resuelve esto poniendo todo elpreprocesamiento dentro del `Pipeline`, de modo que la regla queda impuesta por construcción y no dependede recordarla.**Decisiones que debes justificar por escrito.**1. **Proporción de test.** Lo habitual es reservar entre 10 % y 30 %.2. **Aleatoria o temporal.** Los datos tienen orden temporal, con las columnas `fecha`, `anio` y   `dia_del_anio`. Una partición aleatoria mezcla días adyacentes entre entrenamiento y test, y dos días   consecutivos son muy parecidos entre sí. El curso menciona `TimeSeriesSplit` justamente para datos con   orden temporal. Elige una opción, arguméntala y reconoce la limitación de la que descartaste.3. **Qué hacer con `fecha`.** Decidir si se descarta, si se usa solo a través de las variables derivadas   que ya existen, o si se transforma. Justificar.**Nota del esqueleto.** Esta decisión no es un trámite. Si eliges partición aleatoria sobre datosdiarios, tu estimación del error será optimista, y esa observación pertenece a la pregunta del análisissobre fuentes de sesgo en el proceso de modelado. Reconocerlo suma; ignorarlo resta.

In [ ]:
# TODO 1.3# - Definir la lista de columnas predictoras y la variable objetivo.# - Decidir el tratamiento de la columna de fecha y documentarlo.# - Ejecutar la partición entrenamiento / test con la semilla fija.# - Reportar el tamaño de cada conjunto y verificar que las distribuciones de y sean comparables.

### 1.4 Definición del protocolo de evaluación**Qué hacer.** Definir, una sola vez y para todo el notebook, tres cosas:1. **El esquema de validación cruzada.** Un objeto `KFold` con K fijo, `shuffle` y `random_state`   explícitos. El curso usa K = 5 o K = 10 como valores habituales, que equilibran el costo con la   calidad de la estimación. Usar el **mismo** objeto en las cuatro búsquedas es lo que permite comparar   los modelos en la sección 6: si cada modelo se evalúa con pliegues distintos, la tabla comparativa no   compara nada.2. **Las métricas.** El enunciado exige RMSE, MAE y R² en los puntos 1 y 4. Conviene definir un   diccionario de `scoring` con las tres y declarar cuál es la métrica principal para el `refit` de   `GridSearchCV`.3. **La métrica que gobierna, declarada antes de entrenar.** Elegirla después, comparando resultados y   quedándose con la que favorece al modelo preferido, es dragado de datos y produce una estimación   optimista.**Detalle técnico que suele fallar.** En scikit-learn las métricas de error se expresan como negativas(`neg_root_mean_squared_error`), porque la convención es que un valor mayor es mejor. Al reportar hay quecambiar el signo. Un RMSE negativo en una tabla es un error de reporte que se detecta de inmediato.

In [ ]:
# TODO 1.4# - Instanciar el objeto de validación cruzada que se reutilizará en todo el notebook.# - Definir el diccionario de scoring con RMSE, MAE y R2.# - Declarar la métrica principal para refit y justificar la elección en texto.

### 1.5 Funciones auxiliares de reporte**Qué hacer.** Escribir dos o tres funciones cortas que se reutilizarán en cada sección:- Una que reciba un objeto de búsqueda ajustado y devuelva una fila con el nombre del modelo, los mejores  hiperparámetros, y la media y la desviación estándar de las tres métricas en validación cruzada.- Una que grafique una curva de validación recibiendo el eje del hiperparámetro y los resultados de  entrenamiento y validación, incluyendo la banda de variabilidad.- Opcionalmente, una que extraiga los coeficientes de un pipeline junto con los nombres de las  características generadas.**Por qué hacerlo ahora.** Sin estas funciones el notebook termina con cuatro bloques de código casiidénticos, la tabla comparativa se arma a mano y aparecen inconsistencias entre secciones. Además, lafunción de extracción de nombres es indispensable en la sección 4, donde hay que decir *qué* variablesanula Lasso, no solo cuántas.---**Pregunta de control de la sección 1.** Si escalas todo el conjunto y después partes en entrenamiento ytest, ¿qué información exactamente ha cruzado la frontera, y por qué la estimación del error resultaoptimista y no pesimista?

In [ ]:
# TODO 1.5# - Función de resumen de resultados de una búsqueda (media y desviación por métrica).# - Función de graficación de curvas de validación con banda de variabilidad.# - Función de extracción de coeficientes y nombres de características desde un pipeline.

---## 2. Actividad 1. Modelo de regresión polinomial con búsqueda de hiperparámetros**Rúbrica: criterio 1, 15 %**El objetivo declarado en el enunciado es analizar cómo varía el desempeño a medida que aumenta lacomplejidad e identificar posibles indicios de sobreajuste. El modelo no es el fin, es el instrumentopara observar ese fenómeno.

### 2.1 Construcción del preprocesamiento**Qué hacer.** Construir un `ColumnTransformer` con al menos dos ramas:- **Rama numérica:** imputación, escalado y generación de características polinomiales. El orden importa  y debe justificarse.- **Rama categórica:** el conjunto tiene `estacion_anio`, `mes` y `sector_viento`, que son texto.  Scikit-learn solo admite valores numéricos, de modo que requieren codificación. Decide si entran al  modelo y justifícalo.**Decisiones que debes justificar por escrito.**1. ¿La expansión polinomial se aplica a todas las variables numéricas o a un subconjunto? El número de   columnas que genera la expansión crece como el combinatorio de (p + d) en d. Con veinte variables   numéricas, el grado 3 ya resulta arriesgado. Este es el argumento del curso y conviene retomarlo.2. ¿`include_bias` en la expansión? El estimador lineal ya tiene intercepto.3. ¿Se codifican las categóricas antes o después de la expansión? Expandir variables ya codificadas   multiplica indicadores binarios entre sí, lo cual produce columnas casi siempre nulas.**Nota del esqueleto.** El escalador debe ser un paso **con nombre propio** dentro del pipeline, porquela actividad 1 pide explorar "diferentes estrategias de escalamiento dentro del pipeline". Eso se lograponiendo el objeto escalador como valor de la rejilla, es decir sustituyendo el paso completo, y nocambiando un parámetro suyo.

In [ ]:
# TODO 2.1# - Identificar columnas numéricas y categóricas de forma programática.# - Construir la rama numérica: imputación -> escalado -> expansión polinomial.# - Construir la rama categórica: imputación -> codificación.# - Ensamblar el ColumnTransformer y verificar que transforma sin error una muestra pequeña.

### 2.2 Definición de la rejilla de hiperparámetros**Qué hacer.** Definir el espacio de búsqueda con dos ejes, tal como pide el enunciado:- **Grado del polinomio.** Un rango que permita observar el fenómeno completo, es decir subajuste, óptimo  y sobreajuste. Si el rango es muy corto no se verá nada; si es muy largo, la búsqueda no termina.- **Estrategia de escalamiento.** Al menos las tres alternativas que menciona el curso: estandarización,  escalado robusto y escalado a rango.**Cómo se nombra un hiperparámetro.** La ruta usa dobles guiones bajos para descender por el pipeline,desde el nombre del paso hasta el parámetro del transformador. Escribir mal esa ruta produce un error deparámetro inválido, que es el fallo más común de esta sección.**Advertencia de costo.** El número total de ajustes es el producto de las combinaciones por el número depliegues, más un ajuste final. El enunciado advierte de forma explícita que la búsqueda puede tomarbastante tiempo. Estima el número de ajustes antes de lanzar la búsqueda y déjalo escrito.

In [ ]:
# TODO 2.2# - Definir el diccionario de la rejilla con el eje de grado y el eje de escalador.# - Calcular e imprimir el número total de ajustes que implica la rejilla.

### 2.3 Ejecución de la búsqueda y reporte de resultados**Qué hacer.** Ejecutar `GridSearchCV` sobre el pipeline con el objeto de validación cruzada definido en1.4, activando `return_train_score` y el cálculo de las tres métricas.**Por qué activar el puntaje de entrenamiento.** Sin él no se puede medir la brecha entre entrenamiento yvalidación, y esa brecha es la firma del sobreajuste. Es el dato que sostiene la interpretación de lasección 3.**Qué reportar, como mínimo.**- Los mejores hiperparámetros encontrados.- RMSE, MAE y R² en validación cruzada, con su media y su desviación estándar.- Una tabla ordenada, derivada de los resultados de la búsqueda, que muestre el desempeño de cada  combinación y no solo de la ganadora. Sin esa tabla no se puede argumentar cómo varía el desempeño con  la complejidad.**Lo que NO se hace aquí.** No se evalúa sobre el conjunto de test. El test se usa una sola vez, en lasección 7. Evaluarlo ahora y volver a ajustar decisiones después lo convierte en un segundo conjunto devalidación, y la estimación final queda inflada.---**Pregunta de control de la sección 2.** ¿Por qué el error de entrenamiento baja de forma monótona con elgrado mientras el de validación tiene un mínimo interior? Explica qué cambia en la familia de funcionesdisponibles al aumentar el grado.

In [ ]:
# TODO 2.3# - Instanciar y ajustar GridSearchCV con return_train_score activado.# - Imprimir los mejores hiperparámetros.# - Construir una tabla con todas las combinaciones, ordenada por la métrica principal,#   con media y desviación estándar de RMSE, MAE y R2, y con el puntaje de entrenamiento.# - Guardar el mejor estimador en una variable para la comparación de la sección 6.

### 2.4 Justificación escrita de las decisiones**Qué escribir.** Un párrafo, no una lista de viñetas, que responda: por qué ese rango de grados, por quéesos escaladores, por qué esa métrica principal, y qué muestra la tabla de resultados sobre la relaciónentre complejidad y desempeño. La rúbrica pide el modelo "justificando las decisiones tomadas", de modoque la mitad del criterio 1 vive en este texto y no en el código.

---## 3. Actividad 2. Curvas de validación**Qué hacer.** Generar la curva de validación del modelo polinomial en función del grado, con dos seriesen la misma figura: el error de entrenamiento y el error de validación cruzada. Cada serie debe mostrarel promedio **y su variabilidad**, es decir una banda de más y menos una desviación estándar entrepliegues. El enunciado lo pide de forma explícita.**Dos caminos posibles.** Usar la función `validation_curve` de scikit-learn, o reutilizar los resultadosque ya produjo la búsqueda de la sección 2.3, que contienen exactamente la misma información. La segundaopción evita reentrenar. Cualquiera es válida si se explica.**Qué hay que identificar en el gráfico.**1. El punto mínimo de la curva de validación, que es la elección del hiperparámetro.2. El grado a partir del cual las dos curvas se separan. Esa separación creciente es la firma del   sobreajuste.3. La región de subajuste, a la izquierda, donde ambos errores son altos y están juntos.**Cómo interpretar, en los términos que pide el enunciado.** El error esperado se descompone en sesgo,varianza y ruido, y la complejidad intercambia los dos primeros. Un grado bajo produce sesgo alto yvarianza baja; un grado alto invierte la relación. La curva de validación es la estimación empírica deese intercambio. La interpretación debe hablar de sesgo y varianza de forma concreta sobre **tus**números, no en abstracto.**Nota del esqueleto.** Un error frecuente es graficar solo la curva de validación. Sin la deentrenamiento no hay brecha que medir, y la afirmación "aquí empieza el sobreajuste" queda sin evidencia.Otro error es que un grado alto dispare la escala del eje vertical y aplaste el resto de la curva. Siocurre, recorta el eje y deja constancia del valor real recortado.---**Pregunta de control de la sección 3.** Si la curva de validación desciende y luego se aplana sin volvera subir dentro del rango explorado, ¿qué conclusión puedes sacar y qué deberías hacer antes de afirmarque no hay sobreajuste?

In [ ]:
# TODO 3# - Obtener errores de entrenamiento y validación por grado, con media y desviación.# - Graficar ambas series con banda de variabilidad, ejes rotulados y leyenda.# - Marcar el mínimo de validación en el gráfico.# - Redactar debajo la interpretación en términos de sesgo, varianza y brecha.

---## 4. Actividad 3. Modelos de regresión lineal regularizados**Rúbrica: criterio 2, 15 %**Mientras la sección 2 controla la complejidad eligiendo el grado, lo cual elimina familias enteras detérminos de una sola vez, la regularización ofrece un control continuo que reduce la magnitud de loscoeficientes sin descartar columnas previamente.

### 4.1 Modelo con penalización L2 (Ridge)**Qué hacer.** Construir un pipeline con el mismo preprocesamiento de la sección 2, pero con Ridge comoestimador, e incluir en la rejilla el parámetro de penalización junto con la estrategia de escalamiento.**Detalles técnicos que el curso señala de forma explícita.**1. **La grilla de penalización se recorre en escala logarítmica.** Lo que importa es el orden de magnitud.   Una grilla lineal concentraría casi todos sus puntos en la región donde la curva es plana.2. **En scikit-learn la intensidad se llama `alpha`,** y corresponde al lambda de las fórmulas. Su valor   por omisión es 1.0, que rara vez resulta apropiado.3. **Escalar antes de penalizar es obligatorio.** La penalización suma coeficientes expresados en unidades   distintas. Sin estandarización previa, una variable medida en unidades pequeñas queda prácticamente   exenta de la penalización. Este punto debe aparecer en tu justificación escrita.4. **El intercepto no se penaliza.**

In [ ]:
# TODO 4.1# - Construir el pipeline con Ridge.# - Definir la rejilla con alpha en escala logarítmica y el eje de escalador.# - Ejecutar la búsqueda con el mismo objeto de validación cruzada de 1.4.# - Reportar mejores hiperparámetros y las tres métricas con media y desviación.

### 4.2 Modelo con penalización L1 (Lasso)**Qué hacer.** Lo mismo que en 4.1, cambiando el estimador.**Advertencia técnica del curso.** Con Lasso conviene aumentar `max_iter`. El descenso por coordenadaspuede no converger dentro de las iteraciones por omisión cuando hay muchas columnas correlacionadas, y lalibrería lo señala mediante una advertencia. Si aparecen advertencias de convergencia, no las ignores: osubes el número de iteraciones, o ajustas el rango de `alpha`, y lo dejas documentado.

In [ ]:
# TODO 4.2# - Construir el pipeline con Lasso y un número de iteraciones suficiente.# - Definir la rejilla y ejecutar la búsqueda.# - Verificar que no queden advertencias de convergencia sin resolver ni documentar.

### 4.3 Camino de coeficientes frente a la penalización**Qué hacer.** Graficar cómo cambia la magnitud de los coeficientes al variar `alpha`, para las dospenalizaciones. Es la evidencia visual de la pregunta del análisis sobre el efecto de la regularizaciónen la magnitud y la estabilidad de los coeficientes.**Qué se debe ver y comentar.** Ridge reduce cada coeficiente de forma gradual sin alcanzar el cero, ydistribuye el efecto entre variables correlacionadas. Lasso sustrae una cantidad fija con independencia dela magnitud, de modo que un coeficiente cuya contribución no compensa el valor de la penalización esllevado hasta cero y permanece allí. En el gráfico de Lasso deben verse líneas que tocan el eje y sequedan ahí.

In [ ]:
# TODO 4.3# - Recorrer un rango logarítmico de alpha ajustando cada penalización.# - Almacenar los coeficientes de cada ajuste.# - Graficar coeficiente contra alpha, con escala logarítmica en el eje horizontal,#   una figura por penalización.

### 4.4 Variables anuladas por Lasso**Qué hacer.** Extraer los coeficientes del mejor modelo Lasso junto con los **nombres** de lascaracterísticas que produce el preprocesamiento, contar cuántas sobreviven de cuántas totales, y listarlas de mayor magnitud absoluta.**Por qué los nombres importan.** La pregunta del análisis cualitativo es "¿qué variables fueronseleccionadas como más relevantes por el modelo Lasso?". Responder "conserva 41 de 153 columnas" nocontesta esa pregunta. Hay que nombrarlas y relacionarlas con el fenómeno físico, es decir con latemperatura máxima del día siguiente.**Advertencia de interpretación que debes incluir.** Entre dos variables fuertemente correlacionadas,Lasso tiende a conservar una y anular la otra, y cuál de las dos se retiene puede variar con la muestra.La variable descartada no carece de relevancia: resulta redundante frente a la retenida. En este conjuntode datos hay grupos de variables muy correlacionados por construcción, como la media, el mínimo y elmáximo de una misma magnitud, de modo que esta advertencia es directamente aplicable.

In [ ]:
# TODO 4.4# - Extraer coeficientes y nombres de características del mejor pipeline Lasso.# - Contar características totales y no nulas.# - Listar las de mayor magnitud absoluta con su signo.# - Comparar esa lista con los coeficientes de mayor magnitud en Ridge.

### 4.5 Comparación entre el modelo sin regularización, Ridge y Lasso**Qué hacer.** Una comparación de tres filas: modelo lineal sin penalizar, Ridge y Lasso, con RMSE, MAE yR² en validación cruzada, la desviación estándar entre pliegues y una medida de la magnitud de loscoeficientes, por ejemplo la norma o el conteo de coeficientes no nulos.**Nota del esqueleto.** Para que la fila "sin regularización" sea comparable, debe usar el mismopreprocesamiento y el mismo esquema de pliegues. Si comparas un modelo lineal simple contra un Ridgesobre características polinomiales, no estás midiendo el efecto de la regularización sino el de laexpansión.---**Pregunta de control de la sección 4.** ¿Qué ocurre exactamente con la penalización si eliminas elescalador del pipeline antes de regularizar? Explica el mecanismo, no solo el resultado.

In [ ]:
# TODO 4.5# - Ajustar el modelo sin penalización con el mismo preprocesamiento y los mismos pliegues.# - Consolidar las tres filas en una tabla con métricas, desviaciones y magnitud de coeficientes.# - Redactar la lectura de la tabla.

---## 5. Actividad 4. Modelo de regresión polinomial regularizado**Rúbrica: criterio 3, 10 %****Qué hacer.** Combinar en un solo pipeline la expansión polinomial con un estimador regularizado, ybuscar de forma **simultánea** tres ejes: el grado del polinomio, el parámetro de penalización y laestrategia de escalamiento.**Por qué buscar los tres a la vez y no por separado.** Con la penalización activa, un grado elevadoresulta admisible, porque la penalización controla la magnitud de los términos añadidos. Las dosdecisiones interactúan entre sí. Buscar primero el grado sin penalizar y después la penalización con esegrado fijo produce un óptimo peor, y es un error conceptual que la rúbrica puede castigar en lajustificación.**Decisión que debes justificar.** Ridge o Lasso para esta combinación. Ambos son válidos según elenunciado. El argumento razonable se apoya en lo observado en la sección 4: Ridge conviene cuando haymuchas variables con efecto pequeño, Lasso cuando pocas variables son realmente relevantes. Elige con baseen tu evidencia, no por preferencia.**Advertencia de costo. Esta es la búsqueda más cara del laboratorio.** Tres ejes multiplicados por lospliegues pueden significar cientos o miles de ajustes, y la expansión polinomial de grado alto sobremuchas columnas es costosa en memoria. Estrategias legítimas: acotar el rango de grados con base en loaprendido en la sección 3, reducir la rejilla de escaladores al que ganó de forma consistente, o usar unabúsqueda aleatoria, que muestrea un número fijo de configuraciones al azar y suele encontrar un valorcomparable con una fracción del costo. Si usas búsqueda aleatoria, justifícalo.**Qué debe discutir el análisis de esta sección.** Si la regularización permite incrementar la complejidadsin que el error de validación se deteriore. Compara el grado óptimo aquí con el grado óptimo de lasección 2. Si la penalización cumple su función, debería tolerar un grado mayor.---**Pregunta de control de la sección 5.** Si el grado óptimo con penalización resulta **igual** al gradoóptimo sin penalización, ¿qué estarías observando? ¿Significa que la regularización no sirvió?

In [ ]:
# TODO 5# - Construir el pipeline que combina expansión polinomial y estimador regularizado.# - Definir la rejilla de tres ejes y estimar el número de ajustes.# - Ejecutar la búsqueda (en rejilla o aleatoria, justificando la elección).# - Reportar mejores hiperparámetros y las tres métricas con media y desviación.# - Comparar el grado óptimo obtenido aquí con el de la sección 2.

---## 6. Actividad 5. Tabla comparativa y selección del mejor modelo**Rúbrica: criterio 4, 5 %****Qué hacer.** Consolidar en una sola tabla los resultados de las búsquedas de las secciones 2, 4 y 5.Una fila por modelo, con al menos estas columnas:| Modelo | Mejores hiperparámetros | RMSE medio CV | Desv. est. RMSE | MAE medio | R² medio | Núm. de características |**El criterio de selección, que es el punto de la actividad.** El enunciado es explícito: hay queconsiderar no solo el valor promedio de la métrica principal, sino también su estabilidad. La dispersiónentre pliegues informa sobre la estabilidad del modelo. Un modelo con RMSE medio ligeramente mejor perocon desviación estándar del doble es una elección peor, porque su desempeño depende de qué partición letoque.**Cómo argumentar la elección final.** Tres dimensiones, en este orden:1. **Desempeño.** La métrica principal en validación cruzada.2. **Estabilidad.** La desviación entre pliegues.3. **Complejidad.** A igualdad razonable de lo anterior, el modelo más simple es preferible: menos   características, menor grado, mayor interpretabilidad y menor costo de implementación.**Nota del esqueleto.** No selecciones mirando el conjunto de test. Todavía no lo has tocado y no debestocarlo hasta la sección 7. Elegir el modelo por su desempeño en test y después reportar ese mismo númerocomo estimación del error de generalización es dragado de datos, e invalida el intervalo de confianza dela sección siguiente.---**Pregunta de control de la sección 6.** ¿Por qué la desviación estándar entre pliegues es informaciónsobre el modelo y no solo ruido de la partición? ¿Qué tendría que pasar para que dos modelos con el mismoRMSE medio tengan dispersiones muy distintas?

In [ ]:
# TODO 6# - Construir el DataFrame comparativo con una fila por modelo.# - Ordenar por la métrica principal y mostrarlo.# - Opcional: gráfico de barras con la media y la barra de error de la desviación.# - Seleccionar el modelo final y guardarlo en una variable.

### 6.1 Argumentación escrita de la selección**Qué escribir.** Un párrafo que nombre el modelo elegido, liste sus hiperparámetros ganadores y expliquela elección en las tres dimensiones anteriores. Si el modelo elegido no es el de mejor media, laexplicación de por qué se prefirió la estabilidad es exactamente lo que la rúbrica quiere ver.

---## 7. Actividad 6. Intervalos de confianza por bootstrapping**Rúbrica: criterio 5, 10 %**Aquí, y solo aquí, se usa el conjunto de test, una sola vez.**Qué hacer, paso a paso.**1. Ajustar el modelo seleccionado sobre **todo** el conjunto de entrenamiento con sus mejores   hiperparámetros. Si guardaste el mejor estimador de la búsqueda, ya viene reajustado.2. Predecir sobre el conjunto de test y calcular RMSE, MAE y R² puntuales. Este es el número que se   reporta como estimación del desempeño.3. Ejecutar el bootstrapping: al menos 500 remuestreos, tal como exige el enunciado.4. Calcular los percentiles 2.5 y 97.5 de cada métrica, que dan el intervalo del 95 %.5. Graficar la distribución de las métricas remuestreadas y la distribución de los errores, es decir de   los residuales.**El mecanismo, y el error más común.** El bootstrapping remuestrea **con reemplazo** las observacionesdel conjunto de test y, sobre cada muestra, recalcula la métrica usando las predicciones ya obtenidas.**No** hay que reentrenar el modelo en cada remuestreo. Reentrenar 500 veces responde otra pregunta,cuesta cientos de veces más y no es lo que pide el enunciado. Lo que se cuantifica es la incertidumbre dela métrica debida al tamaño finito del conjunto de test.**Detalle que debes cuidar.** Cada muestra bootstrap debe tener el mismo tamaño que el conjunto de testoriginal, y hay que remuestrear índices, de forma que el valor real y su predicción viajen juntos. Siremuestreas por separado los reales y los predichos, el resultado no significa nada.**Qué discutir en el texto.** La amplitud del intervalo en relación con el valor puntual. Un intervaloestrecho sugiere estabilidad; uno amplio indica que el desempeño reportado depende fuertemente de quéobservaciones cayeron en test. Relaciona la amplitud con el tamaño del conjunto de test: másobservaciones producen intervalos más estrechos.**Sobre la distribución de residuales.** Revisa si está centrada en cero, si es simétrica y si tiene colaspesadas. El cociente entre RMSE y MAE es un indicador útil, porque se aparta de 1 cuando la distribuciónde los errores tiene colas pesadas. Ese cociente conecta directamente con la pregunta sobre laconfiabilidad de las predicciones.---**Pregunta de control de la sección 7.** ¿Por qué el bootstrapping se hace sobre el conjunto de test y nosobre el de entrenamiento? ¿Qué estimaría el intervalo si lo calcularas sobre entrenamiento?

In [ ]:
# TODO 7# - Verificar que el modelo final esté ajustado sobre todo el entrenamiento.# - Predecir sobre test y calcular RMSE, MAE y R2 puntuales.# - Implementar el bucle de bootstrapping con al menos 500 remuestreos sobre índices.# - Calcular percentiles 2.5 y 97.5 para cada métrica.# - Graficar los histogramas de las métricas remuestreadas con el intervalo marcado.# - Graficar la distribución de residuales y calcular el cociente RMSE/MAE.

---## 8. Análisis de resultados**Rúbrica: criterio 6, 35 %. Es el componente de mayor peso del laboratorio.****Cómo responder.** Cada respuesta debe apoyarse en un número o una figura producidos en las seccionesanteriores, citando de dónde sale. Las respuestas correctas pero genéricas, que podrían haberse escritosin haber corrido el notebook, pierden la mayor parte de este 35 %. Una extensión razonable está entre unoy dos párrafos por pregunta.

### 8.1 Análisis cuantitativo**a. ¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?**Apóyate en la sección 7. Cuidado: solo evaluaste en test el modelo seleccionado. Si quieres compararvarios en test, debes decidirlo de antemano y reconocer que cada evaluación adicional erosiona laindependencia del conjunto.**b. ¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada? Si no coincide,¿cuál puede ser la explicación?**Posibles explicaciones que debes contrastar con tus datos: el test es una sola partición y su estimacióntiene más varianza que un promedio de K mediciones; el modelo final se reajusta sobre más datos que encada pliegue; y la selección del hiperparámetro introduce un sesgo optimista en la métrica de validación.**c. ¿El modelo con mejor métrica promedio es necesariamente el más adecuado?**Retoma la discusión de estabilidad de la sección 6 con tus desviaciones estándar concretas.**d. Con base en las curvas de validación, ¿cómo cambia el error al aumentar la complejidad? ¿En qué puntose evidencia sobreajuste?**Cita el grado del mínimo y el grado donde se abre la brecha, con números de tu figura de la sección 3.**e. ¿Cómo afecta la regularización la magnitud y la estabilidad de los coeficientes?**Apóyate en los caminos de coeficientes de la sección 4.3 y en la comparación de magnitudes de 4.5.**f. ¿Los intervalos de confianza sugieren estabilidad o alta variabilidad? ¿Qué implicaciones tiene?**Usa la amplitud relativa al valor puntual, no solo los extremos. Traduce la implicación al caso: si elintervalo del RMSE abarca un rango de varios grados centígrados, ¿qué significa eso para quien use lapredicción?

### 8.2 Análisis cualitativo**a. ¿Qué variables seleccionó Lasso como más relevantes?**Nombres concretos de la sección 4.4, con su signo, y una lectura física de por qué tiene sentido quepredigan la temperatura máxima del día siguiente. Incluye la advertencia sobre variables correlacionadas.**b. ¿Qué interpretación práctica tienen los coeficientes del modelo final?**Cuidado con dos trampas. Primera: si las variables están escaladas, un coeficiente se interpreta pordesviación estándar y no en unidades originales. Segunda: si el modelo es polinomial, los coeficientes delos términos cruzados y cuadráticos no admiten una lectura marginal directa. Si tu modelo final espolinomial, reconoce esa limitación en lugar de forzar una interpretación inválida.**c. ¿Existen diferencias relevantes entre el modelo más preciso y el más interpretable?**Cuantifica el intercambio: cuánto RMSE cuesta pasar del más preciso al más interpretable, y cuántascaracterísticas se ahorran.**d. ¿Qué decisiones estratégicas podría tomar AlpesPlanck a partir de los resultados?**Deben ser decisiones derivadas de tu evidencia, por ejemplo sobre qué variables vale la pena seguirmidiendo, qué precisión es alcanzable, o si conviene recolectar más datos.**e. ¿Mayor precisión implica necesariamente mayor valor para la organización?****f. ¿Un modelo más complejo genera necesariamente mayor valor empresarial?**El enunciado pide discutir interpretabilidad, estabilidad y costo de implementación. Las tres, no una.

### 8.3 Reflexión conceptual**a. ¿Qué relación observas entre complejidad, capacidad de generalización y estabilidad?**Es la síntesis del laboratorio. Debe articular las tres secciones: curva de validación, regularización eintervalos de confianza.**b. ¿Qué fuentes de sesgo podrían estar presentes en los datos o en el proceso de modelado?**Piensa en al menos tres frentes: sesgo en los datos, dado que provienen de una sola estaciónmeteorológica y de un periodo específico; sesgo en el protocolo, si la partición aleatoria rompe laestructura temporal, punto que quedó abierto en la sección 1.3; y sesgo de selección, por elegirhiperparámetros y reportar la misma métrica que los eligió.**c. Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad?**Relaciónalo con dos cosas concretas: la amplitud del intervalo bootstrap, que depende del tamaño delconjunto de test, y la dispersión entre pliegues.

---## 9. Actividad 7. Video explicativo**Rúbrica: criterio 7, 5 %. Máximo 3 minutos.****Audiencia.** El ingeniero de machine learning que lidera el grupo de AlpesPlanck. Es una audienciatécnica: no hay que explicar qué es la validación cruzada, hay que explicar qué decidiste y por qué.**Estructura sugerida para tres minutos.**1. Punto de partida y objetivo, con el resultado del Laboratorio 1 como referencia (unos 20 segundos).2. Los modelos comparados y el criterio de comparación (40 segundos).3. El hallazgo sobre complejidad y sobreajuste, mostrando la curva de validación (40 segundos).4. El modelo seleccionado, sus hiperparámetros y por qué se eligió, incluyendo la estabilidad (40 segundos).5. El intervalo de confianza y qué significa para el uso de la predicción (30 segundos).6. Limitaciones y siguiente paso recomendado (10 segundos).**Error frecuente.** Gastar dos minutos describiendo el código. El líder técnico quiere las decisiones yla evidencia, no un recorrido por las celdas.

---## 10. Uso de herramientas de IA generativa**Rúbrica: criterio 8, 5 %. La sección debe llevar exactamente este título.**Según las reglas del curso, esta sección debe contener cuatro componentes. Si falta cualquiera de ellos,el criterio se pierde completo, que es una forma barata de perder 5 %.**1. Declaración del uso.** Nombre de la herramienta y tipo de uso: ayuda conceptual, generación inicialde código, explicación teórica, depuración, redacción, u otro.**2. Prompts utilizados.** De forma textual o resumida. No hacen falta las interacciones menores, pero sílas que influyeron directamente en el resultado entregado.**3. Análisis crítico del resultado.** Hay que responder al menos dos de estas preguntas:- ¿Qué partes del contenido generado fueron correctas y útiles?- ¿Qué errores, imprecisiones o limitaciones se identificaron?- ¿Qué decisiones técnicas fueron modificadas respecto a la respuesta de la IA y por qué?- ¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?**4. Aportes propios.** Qué fue desarrollado, modificado o decidido por ustedes; qué ajustes se hicieronsobre el código o la explicación original; y qué aprendizajes se obtuvieron del proceso.**Nota del esqueleto.** Si usaste este esqueleto, es un uso de IA generativa y debe declararse. Elanálisis crítico correspondiente puede apoyarse en las decisiones que el esqueleto dejó abiertas de formadeliberada y que ustedes tuvieron que resolver, por ejemplo el tipo de partición frente a la estructuratemporal de los datos, o la elección entre Ridge y Lasso para el modelo polinomial regularizado.

---## 11. Lista de verificación antes de entregar**Contenido técnico**- [ ] Todas las celdas ejecutadas en orden desde un kernel limpio, con las salidas visibles.- [ ] Semilla fija en particiones, validación cruzada y bootstrapping.- [ ] El mismo objeto de validación cruzada en las cuatro búsquedas.- [ ] Todo el preprocesamiento dentro de los pipelines, sin transformaciones previas a la partición.- [ ] RMSE, MAE y R² reportados con el signo correcto en todos los modelos.- [ ] El conjunto de test usado una sola vez, en la sección 7.- [ ] Al menos 500 remuestreos en el bootstrapping.- [ ] Sin advertencias de convergencia sin resolver ni documentar.**Documentación**- [ ] Cada decisión metodológica justificada por escrito, no solo implementada.- [ ] Las quince preguntas del análisis de resultados respondidas y ancladas a números o figuras propios.- [ ] Todas las figuras con título, ejes rotulados y leyenda.- [ ] Notas del esqueleto eliminadas.**Entrega**- [ ] Archivos `.ipynb` y `.html` con los nombres de los estudiantes.- [ ] Video de máximo 3 minutos.- [ ] Sección "Uso de herramientas de IA generativa" con los cuatro componentes.- [ ] Los dos integrantes que presentan registrados en el grupo GL2 para habilitar el enlace.- [ ] Entrega antes del 14 de septiembre a las 20:00. Entre esa hora y las 2:00 del 15 de septiembre la      entrega se califica sobre 3.5; después de ese momento, sobre 0.